# Metajeux-06 : Le chemin minimal sur un second substrat — jeux ordinaux 3×2, murs traversables et raccourcis

**Navigation** : [<< Metajeux-05-Temoins-Impossibilite](Metajeux-05-Temoins-Impossibilite-Python.ipynb) | [Version Python](Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb) | [Index](README.md)

**Kernel** : .NET C#

***

> **Seconde attestation, second substrat** : la première ([Metajeux-04](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb), ex. 1-4, et [Metajeux-05](Metajeux-05-Temoins-Impossibilite-Python.ipynb), ex. A-C) vivait sur les jeux 2×2 ordinaux de Robinson-Goforth ; celle-ci monte au substrat **3×2 ordinal** — chaque joueur range 6 cases, l'univers des chambres strictes passe de 24×24 = 576 à 720×720 = 518 400 jeux.

**La question falsifiable que ce notebook pose.** Sur 2×2, GT-20 a mesuré que traverser les murs (passer par des ex æquo) ne raccourcit **jamais** un chemin minimal entre chambres : 0 raccourci sur 576 paires. Ce verdict survive-t-il au changement de substrat ? La réponse mesurée ici est **non** : sur 3×2, 77 760 paires sur 518 400 (**15,0 %**) raccourcissent en traversant les murs, et le diamètre par côté se contracte de 15 à 10. La stratification qui ne comprimait pas la métrique à 2×2 **devient une compression métrique** à 3×2.

**La loi que ce notebook met à l'épreuve** — Loi II du deuxième voyage de digestion ICT :

> *recoordonner + passer du vérificateur au constructeur* : le précédent important n'est pas seulement le changement de coordonnées ; c'est changement de coordonnées **plus** passage du vérificateur au constructeur.

Comme GT-20, chaque témoin est produit par un **constructeur** (BFS + remontée des parents) puis **certifié par un vérificateur séparé** qui ne réutilise rien : il re-dérive l'adjacence par son propre code et re-fait une recherche exhaustive pour la minimalité.

**Plan.** §1 les deux univers (ordres faibles, chambres) ; §2 les générateurs et les trois modèles de graphe ; §3 reproduction du verdict 2×2 sous modèle fermé ; §4 le recensement exhaustif 3×2 — la réfutation ; §5 le produit de jeux et son lemme de décomposition ; §6 le constructeur et le vérificateur séparés (Loi II), y compris le chemin de l'antipode **à travers les murs** ; §7 le contrôle négatif ; §8 les géodésiques ; §9 trois exercices.

In [1]:
// === Configuration : GameTheory 20c, chemin minimal 3x2 ordinal - seconde attestation de l'operation 13 ===
using System;
using System.Collections.Generic;
using System.Linq;
using System.Globalization;
using System.Diagnostics;

// Les tables sont des int[] : les dictionnaires BCL n'ont pas d'egalite de valeur sur les tableaux,
// la cle join(",") la fournit (equiv tuple Python).
static string Key(int[] t) => string.Join(",", t);

// Affichage d'une table au format tuple de la source Python : (1, 2, 3, 4, 5, 6).
static string Tst(int[] t) => "(" + string.Join(", ", t) + ")";

Console.WriteLine("GameTheory 20c : chemin minimal sur jeux ordinaux 3x2 (seconde attestation, operation 13)");
Console.WriteLine("Representation : jeu = (table_ligne, table_colonne), chaque table range 6 cases, rangs {1..6}");
Console.WriteLine("Convention GT-20 conservee : le rang 6 est le MEILLEUR resultat possible");
Console.WriteLine("Loi II : le constructeur PRODUIT le chemin, le verificateur separe le CERTIFIE par re-derivation");

The below script needs to be able to find the current output cell; this is an easy method to get it.

GameTheory 20c : chemin minimal sur jeux ordinaux 3x2 (seconde attestation, operation 13)


Representation : jeu = (table_ligne, table_colonne), chaque table range 6 cases, rangs {1..6}


Convention GT-20 conservee : le rang 6 est le MEILLEUR resultat possible


Loi II : le constructeur PRODUIT le chemin, le verificateur separe le CERTIFIE par re-derivation


### Lecture du résultat — le changement d'échelle comme variable expérimentale : un seul paramètre change

Le changement d'échelle est la variable expérimentale : GT-20 étudiait des tables 2×2 (4 cases par joueur), ici chaque joueur ordinal dispose d'une table **3×2** — 6 cases à ranger. Toute la machinerie conceptuelle (chambres strictes, murs d'ex æquo, facettes) est portée telle quelle ; c'est précisément ce qui rend la comparaison falsifiable : *un seul paramètre change, le substrat*. La convention des rangs suit GT-20 (`6` = meilleur), et la Loi II structure tout le notebook : aucun témoin n'est accepté sur la parole du code qui l'a produit.

In [2]:
// === Section 1.1 : enumeration des ordres faibles sur 6 cases (et rappel 4 cases) ===

// Cle canonique d'un uplet : relabelage croissant des valeurs distinctes (GT-20, portee tel quel).
static int[] Canonique(int[] t)
{
    var vals = t.Distinct().OrderBy(v => v).ToList();
    return t.Select(v => vals.IndexOf(v) + 1).ToArray();
}

// Ordre lexicographique sur les tables (equiv comparaison de tuples Python).
static int CmpLex(int[] a, int[] b)
{
    var n = Math.Min(a.Length, b.Length);
    for (int i = 0; i < n; i++)
    {
        var c = a[i].CompareTo(b[i]);
        if (c != 0) return c;
    }
    return a.Length.CompareTo(b.Length);
}

// Produit cartesien {1..n}^k en ordre lexicographique (equiv itertools.product).
static IEnumerable<int[]> Produit(int n, int k)
{
    if (k == 0) { yield return Array.Empty<int>(); yield break; }
    foreach (var queue in Produit(n, k - 1))
        for (var v = 1; v <= n; v++)
        {
            var r = new int[k];
            r[0] = v;
            Array.Copy(queue, 0, r, 1, k - 1);
            yield return r;
        }
}

// Tous les rangements faibles (avec ex aequo) de n cases, formes canoniques, tries.
static List<int[]> OrdresFaibles(int n)
{
    var vus = new HashSet<string>();
    var res = new List<int[]>();
    foreach (var t in Produit(n, n))
    {
        var c = Canonique(t);
        if (vus.Add(Key(c))) res.Add(c);
    }
    res.Sort(Comparer<int[]>.Create(CmpLex));
    return res;
}

var OF4 = OrdresFaibles(4);
var ST4 = OF4.Where(t => t.Distinct().Count() == 4).ToList();
var OF6 = OrdresFaibles(6);
var ST6 = OF6.Where(t => t.Distinct().Count() == 6).ToList();

Console.WriteLine($"Substrat 2x2 (GT-20)  : ordres faibles = {OF4.Count} (nombre de Fubini F4) | stricts = {ST4.Count} | chambres = {ST4.Count * ST4.Count}");
Console.WriteLine($"Substrat 3x2 (ici)    : ordres faibles = {OF6.Count} (nombre de Fubini F6) | stricts = {ST6.Count} | chambres = {ST6.Count * ST6.Count}");
Console.WriteLine($"Avec ex aequo 3x2     : {OF6.Count - ST6.Count} tables liees -- les futures faces des murs");

Substrat 2x2 (GT-20)  : ordres faibles = 75 (nombre de Fubini F4) | stricts = 24 | chambres = 576


Substrat 3x2 (ici)    : ordres faibles = 4683 (nombre de Fubini F6) | stricts = 720 | chambres = 518400


Avec ex aequo 3x2     : 3963 tables liees -- les futures faces des murs


### Lecture du résultat — les nombres de Fubini : 720 tables strictes, 4 683 ordres faibles, 518 400 chambres

Les nombres de Fubini comptent les ordres faibles : F4 = 75, F6 = 4 683. Le côté 3×2 porte donc 720 ordres stricts (= 6!) et 4 683 ordres faibles, contre 24 (= 4!) et 75 pour le 2×2. Le produit des deux joueurs donne l'univers des jeux : 518 400 chambres strictes en 3×2, 21 930 489 jeux au total si l'on admet les murs des deux côtés à la fois — un univers que nous ne parcourrons pas exhaustivement au niveau jeu (voir le lemme du §5, qui le couvre par décomposition). L'explosion combinatoire est le prix du changement de substrat ; elle est aussi ce qui rend la question des raccourcis non triviale.

In [3]:
// === Section 1.2 : les jeux canoniques 3x2 (convention GT-21 / GT-3b / GT-20 etendue) ===

static void AfficherJeu(string nom, (int[] Ligne, int[] Colonne) jeu)
{
    Console.WriteLine(nom);
    for (int i = 0; i < 3; i++)
        Console.WriteLine($"  Ligne | {jeu.Ligne[2 * i],2}  {jeu.Ligne[2 * i + 1],-2}|      Colonne | {jeu.Colonne[2 * i],2}  {jeu.Colonne[2 * i + 1],-2}|");
}

var IDENTITE = (Ligne: new[] { 1, 2, 3, 4, 5, 6 }, Colonne: new[] { 1, 2, 3, 4, 5, 6 });   // les deux joueurs preferent la meme case
var RENVERSE = (Ligne: new[] { 6, 5, 4, 3, 2, 1 }, Colonne: new[] { 6, 5, 4, 3, 2, 1 });   // les deux antipodes du permutoedre S6
var ROTATION  = (Ligne: new[] { 3, 4, 5, 6, 1, 2 }, Colonne: new[] { 5, 6, 1, 2, 3, 4 });  // conflit pur : chaque meilleure case d'un joueur
var MUR_PARTIEL = (Ligne: new[] { 1, 2, 2, 3, 4, 5 }, Colonne: new[] { 1, 2, 3, 4, 5, 6 }); // Ligne indifferente entre ses cases 2 et 3

foreach (var (nom, j) in new[] { ("Identite", IDENTITE), ("Renversement", RENVERSE), ("Rotation", ROTATION),
                                 ("Mur partiel (Ligne indifferente 2~3)", MUR_PARTIEL) })
{
    AfficherJeu(nom, j);
    bool strict = j.Ligne.Distinct().Count() == 6 && j.Colonne.Distinct().Count() == 6;
    Console.WriteLine($"  strict (chambre) : {strict} | mur : {!strict}");
    Console.WriteLine();
}

Identite


  Ligne |  1  2 |      Colonne |  1  2 |


  Ligne |  3  4 |      Colonne |  3  4 |


  Ligne |  5  6 |      Colonne |  5  6 |


  strict (chambre) : True | mur : False


Renversement


... (10 outputs left out)

  Ligne |  1  2 |      Colonne |  1  2 |


  Ligne |  2  3 |      Colonne |  3  4 |


  Ligne |  4  5 |      Colonne |  5  6 |


  strict (chambre) : False | mur : True


### Lecture du résultat — les jeux canoniques : deux antipodes, un conflit d'intérêts, et `MUR_PARTIEL` hors des chambres

L'`Identite` et le `Renversement` sont les deux antipodes du permutoèdre S6 × S6 : leur distance par côté est le diamètre (§2.2). `ROTATION` donne un exemple de conflit d'intérêts ordinal pur — les meilleures cases des deux joueurs diffèrent. `MUR_PARTIEL` n'est **pas** une chambre : la table de Ligne porte un ex æquo (deux cases au rang 2), c'est un point de l'univers étendu, une face du mur. La question de l'opération 13 est exactement : que se passe-t-il quand un chemin **traverse** de telles faces ?

Un mot sur ce que « jeu ordinal 3×2 » veut dire ici, pour éviter tout malentendu de modélisation. Chaque joueur range les six cases de la matrice 3 × 2 (trois rangées, deux colonnes) par ordre de préférence, sans intensité cardinale : seuls les rangs comptent, exactement comme dans le 2×2 de Robinson-Goforth porté par GT-20/GT-3b. Les 720 tables strictes par côté sont les 6! ordres totaux possibles ; une table avec ex æquo est un ordre *partiel* au sens où plusieurs cases partagent un rang. Aucune théorie des équilibres n'est mobilisée dans ce notebook : l'objet d'étude est la **géométrie de l'espace des préférences** — la question de l'opération 13 est une question de métrique sur cet espace, pas de comportement stratégique.

In [4]:
// === Section 2.1 : les generateurs -- swap de facettes, fusion (descente de mur), scission (remontee) ===

// Traversee de facette k<->k+1 : echange les cases portant les valeurs k et k+1 (tables strictes).
static int[] SwapAdjacent(int[] t, int k)
{
    int pk = Array.IndexOf(t, k), pk1 = Array.IndexOf(t, k + 1);
    var l = (int[])t.Clone();
    (l[pk], l[pk1]) = (l[pk1], l[pk]);
    return l;
}

// Descente de mur (le faire_le_tie de GT-20) : coalesce les valeurs k et k+1, forme canonique.
static int[] Fusion(int[] t, int k)
{
    return Canonique(t.Select(v => v == k + 1 ? k : v).ToArray());
}

// Sous-ensembles de taille j d'une liste d'indices, en ordre lexicographique (equiv itertools.combinations).
static IEnumerable<List<int>> Combinaisons(List<int> P, int j)
{
    if (j == 0) { yield return new List<int>(); yield break; }
    for (int start = 0; start <= P.Count - j; start++)
        foreach (var tail in Combinaisons(P.Skip(start + 1).ToList(), j - 1))
        {
            var res = new List<int> { P[start] };
            res.AddRange(tail);
            yield return res;
        }
}

// Remontee de mur : chaque bipartition non triviale d'un bloc d'ex aequo, dans les deux ordres.
static List<int[]> Scissions(int[] t)
{
    var res = new List<int[]>();
    foreach (var v in t.Distinct().OrderBy(x => x))
    {
        var P = new List<int>();
        for (int i = 0; i < t.Length; i++) if (t[i] == v) P.Add(i);
        int m = P.Count;
        if (m < 2) continue;
        for (int j = 1; j < m; j++)
            foreach (var S in Combinaisons(P, j))
            {
                var s = S.ToHashSet();
                var nouv = new int[t.Length];
                for (int i = 0; i < t.Length; i++)
                {
                    int x = t[i];
                    nouv[i] = x == v && !s.Contains(i) ? v + 1
                            : x == v ? v
                            : x > v ? x + 1 : x;
                }
                res.Add(Canonique(nouv));
            }
    }
    return res;
}

var t = new[] { 1, 2, 3, 4, 5, 6 };
Console.WriteLine($"Table identite                     : {Tst(t)}");
Console.WriteLine($"swap facette 2<->3                 : {Tst(SwapAdjacent(t, 2))}");
Console.WriteLine($"fusion 2~3 (descente de mur)       : {Tst(Fusion(t, 2))}");
Console.WriteLine($"scissions de la table liee (1,2,2,3,4,5) : [{string.Join(", ", Scissions(new[] { 1, 2, 2, 3, 4, 5 }).Select(Tst))}]");
Console.WriteLine($"Le swap de singletons = fusion puis scission dans l'autre sens : {Scissions(Fusion(t, 2)).Any(u => Key(u) == Key(SwapAdjacent(t, 2)))}");

Table identite                     : (1, 2, 3, 4, 5, 6)


swap facette 2<->3                 : (1, 3, 2, 4, 5, 6)


fusion 2~3 (descente de mur)       : (1, 2, 2, 3, 4, 5)


scissions de la table liee (1,2,2,3,4,5) : [(1, 2, 3, 4, 5, 6), (1, 3, 2, 4, 5, 6)]


Le swap de singletons = fusion puis scission dans l'autre sens : True


### Lecture du résultat — trois familles de générateurs, et le swap décomposé en fusion puis scission

Trois familles de générateurs, et une identité structurelle qui éclaire tout le notebook : **un swap de facettes entre deux valeurs strictes est exactement une fusion suivie d'une scission dans l'ordre opposé** (vérifié ci-dessus). Autrement dit, le permutoèdre des chambres est *plongé* dans le graphe des ordres faibles : chaque arête de chambre se décompose en deux pas de mur. C'est cette décomposition qui rend la question des raccourcis sensée — traverser les murs coûte des pas *supplémentaires* pour réordonner, mais permet de déplacer des **blocs entiers** en une seule scission. Le verdict 2×2 de GT-20 disait que ce troc n'était jamais rentable à 4 cases ; le §4 mesure s'il le reste à 6.

**Les trois modèles de graphe utilisés dans la suite** (déclarés ici, utilisés ensuite) :

| Modèle | Sommets | Arêtes | Question |
|---|---|---|---|
| **Chambres** (permutoèdre) | ordres stricts | swaps de facettes | distance de référence `d_perm` |
| **Murs dirigés** (GT-20 historique) | ordres faibles | swaps + fusions *à sens unique* | reproduction du cadre original |
| **Treillis fermé** (canonique) | ordres faibles | fusions + scissions, non dirigé | un mur se traverse **dans les deux sens** |

Le modèle fermé est celui du graphe de couverture du treillis des ordres faibles — l'objet standard de la combinatoire des arrangements de type A. Il est le seul où « traverser un mur » a un sens honnête : descendre une facette *et pouvoir remonter ailleurs*. C'est lui qui porte la question falsifiable.

Pourquoi ce choix canonique plutôt qu'un autre ? Une modélisation naïve du modèle fermé consisterait à garder les swaps *aussi* sur les tables liées — mais un swap entre deux valeurs liées n'a pas de définition naturelle (échanger quelles occurrences ?), et l'échange de blocs entiers déplace plusieurs cases pour un pas, ce qui fabrique des raccourcis par fiat de modélisation plutôt que par la structure. En restant sur les seules relations de couverture du treillis — fusionner deux blocs adjacents, ou scinder un bloc — chaque pas a exactement une signification : on descend dans la stratification (on renonce à départager deux cases) ou on remonte (on les départage autrement). La question « les murs raccourcissent-ils ? » devient alors une question sur l'objet mathématique standard, pas sur les détails d'une implémentation.

In [5]:
// === Section 2.2 : construction des adjacences, symetrie, connectivite, diametres mesures ===

static List<int[]> VoisinsPermutoedre(int[] t, int n)
{
    var res = new List<int[]>();
    for (int k = 1; k < n; k++) res.Add(SwapAdjacent(t, k));
    return res;
}

// Fusions (k et k+1 valeurs presentes) + scissions -- le graphe de couverture du treillis.
static List<int[]> VoisinsTreillis(int[] t, int n)
{
    var res = new List<int[]>();
    var vals = t.Distinct().ToHashSet();
    for (int k = 1; k < n; k++)
        if (vals.Contains(k) && vals.Contains(k + 1))
            res.Add(Fusion(t, k));
    res.AddRange(Scissions(t));
    return res;
}

// BFS exhaustive ; adj est un dictionnaire cle -> voisinage (le constructeur l'utilisera aussi).
static Dictionary<string, int> Bfs(int[] src, Dictionary<string, List<int[]>> adj)
{
    var dist = new Dictionary<string, int> { [Key(src)] = 0 };
    var q = new Queue<int[]>(new[] { src });
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        foreach (var v in adj[Key(u)])
        {
            var kv = Key(v);
            if (!dist.ContainsKey(kv))
            {
                dist[kv] = dist[Key(u)] + 1;
                q.Enqueue(v);
            }
        }
    }
    return dist;
}

var graphs = new Dictionary<int, (Dictionary<string, List<int[]>> AdjP, Dictionary<string, List<int[]>> AdjC)>();
foreach (var (n, OF, ST) in new[] { (4, OF4, ST4), (6, OF6, ST6) })
{
    var adjP = ST.ToDictionary(t => Key(t), t => VoisinsPermutoedre(t, n));
    var adjC = OF.ToDictionary(t => Key(t), t => VoisinsTreillis(t, n));
    long asym = 0;
    foreach (var (ku, vs) in adjC)
        foreach (var v in vs)
            if (!adjC[Key(v)].Any(w => Key(w) == ku)) asym++;
    double degMoy = adjC.Values.Sum(v => v.Count) / (double)adjC.Count;
    var dP = Bfs(ST[0], adjP);
    var dC = Bfs(OF[0], adjC);
    graphs[n] = (adjP, adjC);
    Console.WriteLine($"n={n} | stricts={ST.Count} faibles={OF.Count} | treillis : asymetries={asym}, degre moyen={degMoy.ToString("F1", CultureInfo.InvariantCulture)}, connexe={dC.Count}/{OF.Count}, excentricite du centre tout-lie={dC.Values.Max()} -> diametre = {2 * dC.Values.Max()}");
    Console.WriteLine($"     | permutoedre : connexe={dP.Count}/{ST.Count}, diametre={dP.Values.Max()} (= inversions de l'element le plus long w0 : {n * (n - 1) / 2})");
}

n=4 | stricts=24 faibles=75 | treillis : asymetries=0, degre moyen=4.2, connexe=75/75, excentricite du centre tout-lie=3 -> diametre = 6


     | permutoedre : connexe=24/24, diametre=6 (= inversions de l'element le plus long w0 : 6)


n=6 | stricts=720 faibles=4683 | treillis : asymetries=0, degre moyen=7.1, connexe=4683/4683, excentricite du centre tout-lie=5 -> diametre = 10


     | permutoedre : connexe=720/720, diametre=15 (= inversions de l'element le plus long w0 : 15)


### Lecture du résultat — les diamètres mesurés : 6 et 15 sur le permutoèdre, 6 et 10 sur le treillis

Deux contrôles structurels passent : le graphe du treillis est **exactement symétrique** (0 asymétrie — la remontée de mur est bien l'inverse de la descente), et les diamètres mesurés du permutoèdre valent 6 (n=4) et 15 (n=6), soit les nombres d'inversions de l'élément le plus long w0 de S4 et S6 — la théorie des groupes de Coxeter prédit ces valeurs et la mesure les confirme.

Pour le treillis, la mesure directe donne l'**excentricité de la table tout-liée** : 3 (n=4) et 5 (n=6). Ce n'est pas le diamètre, mais presque : la table tout-liée est le **centre** du treillis — atteindre une table à r blocs depuis elle demande exactement r−1 scissions, une par bloc créé — donc toute paire de sommets est à au plus 2×(n−1) l'un de l'autre en passant par le centre. Le diamètre du treillis vaut ainsi 6 (n=4) et 10 (n=6), et la borne est *serrée* : le §6 la réalise entre les deux antipodes stricts, certifié par le vérificateur. Premier fait saillant déjà visible à n=6 : 10 < 15 — la contraction métrique existe dans l'univers étendu ; reste à savoir si elle touche les paires de *chambres* (le §4 y répond). À n=4 en revanche, 6 = 6 : premier indice que le 2×2 n'a pas de marge.

In [6]:
// === Section 3 : reproduction du verdict 2x2 de GT-20 sous le modele ferme ===
// GT-20 avait mesure 0 raccourci sur les 576 paires de chambres 2x2, mais avec des murs a sens
// unique (swaps + faire_le_tie diriges) : une fois un mur descendu, impossible de remonter, donc
// le 0 etait en partie structurel. On re-meure ici sous le modele ferme (fusions + scissions).

// Pour chaque paire ordonnee de chambres : d_treillis < d_permutoedre ?
static (int Paires, int Raccourcis, int Egal, (int[] A, int[] B, int Dperm, int Dtl) Exemple) Recensement(
    int n, List<int[]> ST, Dictionary<string, List<int[]>> adjP, Dictionary<string, List<int[]>> adjC)
{
    int raccourcis = 0, egal = 0, paires = 0;
    (int[], int[], int, int)? exemple = null;
    foreach (var a in ST)
    {
        var dperm = Bfs(a, adjP);
        var dtl = Bfs(a, adjC);
        foreach (var b in ST)
        {
            paires++;
            if (dtl[Key(b)] < dperm[Key(b)])
            {
                raccourcis++;
                if (exemple is null && CmpLex(b, a) > 0)
                    exemple = (a, b, dperm[Key(b)], dtl[Key(b)]);
            }
            else if (dtl[Key(b)] == dperm[Key(b)]) egal++;
        }
    }
    return (paires, raccourcis, egal, exemple ?? (null, null, 0, 0));
}

var sw = Stopwatch.StartNew();
var (paires4, rc4, eg4, ex4) = Recensement(4, ST4, graphs[4].AdjP, graphs[4].AdjC);
Console.WriteLine($"Recensement 2x2 (modele ferme) : {paires4} paires ordonnees");
Console.WriteLine($"  raccourcis (d_treillis < d_perm) : {rc4}");
Console.WriteLine($"  egalites (aucun gain a traverser) : {eg4}");
Console.WriteLine($"  exemple de raccourci : {(ex4.A is null ? "None" : $"{Tst(ex4.A)} -> {Tst(ex4.B)} : d_perm = {ex4.Dperm}, d_treillis = {ex4.Dtl}")}");
Console.WriteLine($"  ({sw.Elapsed.TotalSeconds.ToString("F1", CultureInfo.InvariantCulture)}s)");

Recensement 2x2 (modele ferme) : 576 paires ordonnees


  raccourcis (d_treillis < d_perm) : 0


  egalites (aucun gain a traverser) : 48


  exemple de raccourci : None


  (0.0s)


### Lecture du résultat — le verdict 2×2 survit au modèle fermé : 0 raccourci sur 576 paires

**Le verdict de GT-20 survit au modèle fermé** : 0 raccourci sur 576 paires, même quand les murs se traversent dans les deux sens. La mesure originale n'était donc pas un artefact du sens unique — à 4 cases, le troc « des pas supplémentaires pour réordonner » n'est jamais rentable, et les 48 égalités (dont l'antipode, 6 = 2×(4−1)) montrent que le meilleur chemin par les murs égale tout au plus le chemin de chambres. La généralisation à tester en §4 est ainsi proprement posée : *le même banc, le même modèle fermé, un seul changement — le substrat.*

In [7]:
// === Section 4 : le recensement exhaustif 3x2 -- la question falsifiable, la reponse mesuree ===

var sw4 = Stopwatch.StartNew();
var (paires6, rc6, eg6, ex6) = Recensement(6, ST6, graphs[6].AdjP, graphs[6].AdjC);
sw4.Stop();
Console.WriteLine($"Recensement 3x2 (modele ferme) : {paires6} paires ordonnees de chambres (720 x 720)");
Console.WriteLine($"  raccourcis (d_treillis < d_perm) : {rc6}  ({(100.0 * rc6 / paires6).ToString("F1", CultureInfo.InvariantCulture)} %)");
Console.WriteLine($"  egalites                         : {eg6}");
Console.WriteLine($"  exemple : {Tst(ex6.A)} -> {Tst(ex6.B)} : d_perm = {ex6.Dperm}, d_treillis = {ex6.Dtl}");
Console.WriteLine($"  diametre permutoedre = 15 | diametre treillis = 10 (centre tout-lie d'excentricite 5, section 2.2)");
Console.WriteLine($"  ({sw4.Elapsed.TotalSeconds.ToString("F1", CultureInfo.InvariantCulture)}s d'execution -- exhaustive, pas d'echantillonnage)");

Recensement 3x2 (modele ferme) : 518400 paires ordonnees de chambres (720 x 720)


  raccourcis (d_treillis < d_perm) : 77760  (15.0 %)


  egalites                         : 65520


  exemple : (1, 2, 3, 4, 5, 6) -> (1, 5, 6, 4, 3, 2) : d_perm = 9, d_treillis = 8


  diametre permutoedre = 15 | diametre treillis = 10 (centre tout-lie d'excentricite 5, section 2.2)


  (6.9s d'execution -- exhaustive, pas d'echantillonnage)


### Lecture du résultat — la généralisation est réfutée : 77 760 paires sur 518 400, diamètre contracté de 15 à 10

**La généralisation est réfutée.** Sur le substrat 3×2, 77 760 paires ordonnées de chambres sur 518 400 — **15,0 %** — ont un chemin plus court en traversant les murs qu'en restant dans les chambres, et le diamètre par côté se contracte de 15 à 10. Le verdict 2×2 (« la stratification ne comprime pas la métrique ») n'était pas un théorème de l'opération : c'était une propriété du petit substrat.

**Pourquoi 10, et pourquoi par là.** Le chemin intégral par les murs — fusionner tout (5 fusions successives jusqu'à la table tout-liée), puis scissionner dans l'ordre cible (5 remontées) — coûte exactement 2×(6−1) = 10 : c'est lui qui réalise le nouveau diamètre entre antipodes, là où le permutoèdre en demande 15. L'économie vient des **blocs** : une seule scission d'un bloc de taille m replace m cases d'un coup, quand le permutoèdre doit les permuter une à une.

**L'arithmétique du troc** mérite d'être écrite noir sur blanc, car c'est elle qui explique la frontière 2×2 / 3×2. Toute traversée utile de mur coûte *au moins* 2 pas (une descente, une remontée) — pour rentabiliser, le déplacement de blocs réalisé entre les deux doit économiser au moins 2 swaps. Or échanger les positions de deux éléments séparés par j rangs coûte **2j - 1** swaps au permutoèdre (chacun des deux éléments passe les j - 1 éléments qui les séparent, plus l'autre : 2(j - 1) + 1), tandis qu'une scission tranche d'un geste toutes les ex æquo entre les deux sections qu'elle sépare. Sur l'antipode, la loi est lisible et mesurable : renverser les n cases coûte C(n, 2) swaps au permutoèdre (6 à n = 4, 15 à n = 6 — §2.2) et 2(n - 1) pas par les murs (6 — §3, et 10 — §6), d'où un gain net de C(n, 2) - 2(n - 1) = (n - 1)(n - 4)/2 : **0 à 4 cases**, **5 à 6 cases**. À 4 cases le troc ne peut donc rien gagner — le recensement du §3 le mesure : jamais. À 6 cases, la table tout-liée offre un bloc de taille 6 : y loger l'ordre renversé ne demande que 5 scissions (chacune détache une case du bloc), là où le permutoèdre exige un swap par paire inversée — d'où le gain net de 5, exactement mesuré au §4. La question de savoir si *toutes* les paires gagnent (non : 85 % des paires ne raccourcissent pas — et parmi elles, la majorité des trajets sont même plus longs par les murs) et de combien est précisément ce que le recensement mesure, et l'exercice 2 y revient par le bout des antipodes.

C'est exactement le type de résultat que la table des opérations [#12204](https://github.com/jsboige/CoursIA/issues/12204) veut pour l'opération 13 : non pas « on rejoue le témoin », mais **le témoin produit une connaissance nouvelle sur la frontière de la loi** — ici, la borne où « traverser un mur » cesse d'être métriquement neutre et devient un raccourci structurel.

In [8]:
// === Section 5 : le produit de jeux et son lemme de decomposition ===
// Les generateurs ne touchent qu'un cote a la fois : le graphe de jeux est le produit cartesien
// des deux graphes par cote. Lemme : d_jeu(G, H) = d_cote(row_G, row_H) + d_cote(col_G, col_H),
// dans CHACUN des deux modeles (chambres et treillis). Verification directe sur l'antipode.

static string CleJeu((int[] Ligne, int[] Colonne) j) => Key(j.Ligne) + "|" + Key(j.Colonne);

// <= 10 voisins : 5 swaps par cote, calcules a la volee (pas d'enumeration prealable).
static List<(int[] Ligne, int[] Colonne)> VoisinsJeuChambres((int[] Ligne, int[] Colonne) jeu, int n = 6)
{
    var res = new List<(int[], int[])>();
    foreach (var cote in new[] { 0, 1 })
    {
        var t = cote == 0 ? jeu.Ligne : jeu.Colonne;
        for (int k = 1; k < n; k++)
        {
            var v = SwapAdjacent(t, k);
            res.Add(cote == 0 ? (v, jeu.Colonne) : (jeu.Ligne, v));
        }
    }
    return res;
}

static Dictionary<string, int> BfsJeu((int[] Ligne, int[] Colonne) src, Func<(int[], int[]), List<(int[], int[])>> voisinsFn)
{
    var dist = new Dictionary<string, int> { [CleJeu(src)] = 0 };
    var q = new Queue<(int[], int[])>(new[] { src });
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        foreach (var v in voisinsFn(u))
        {
            var kv = CleJeu(v);
            if (!dist.ContainsKey(kv))
            {
                dist[kv] = dist[CleJeu(u)] + 1;
                q.Enqueue(v);
            }
        }
    }
    return dist;
}

var sw5 = Stopwatch.StartNew();
var distJeu = BfsJeu(IDENTITE, j => VoisinsJeuChambres(j));
int atteint = distJeu.Count;
int dDirecte = distJeu[CleJeu(RENVERSE)];
int dDecomposee = 15 + 15;
Console.WriteLine($"BFS directe sur le produit des 518 400 chambres : {atteint} jeux atteints depuis Identite");
Console.WriteLine($"distance Identite -> Renversement : directe = {dDirecte} | par decomposition = {dDecomposee}");
Console.WriteLine($"concordance : {dDirecte == dDecomposee}  ({sw5.Elapsed.TotalSeconds.ToString("F1", CultureInfo.InvariantCulture)}s)")

BFS directe sur le produit des 518 400 chambres : 518400 jeux atteints depuis Identite


distance Identite -> Renversement : directe = 30 | par decomposition = 30


concordance : True  (2.8s)


### Lecture du résultat — le lemme de décomposition contrôlé par BFS : 30 = 15 + 15

La BFS directe sur le produit des chambres (518 400 sommets, degré 10) confirme le lemme de décomposition : 30 = 15 + 15. Ce lemme est ce qui garde le substrat 3×2 **tractable honnêtement** : l'univers complet de jeux avec murs des deux côtés compte 4 683² ≈ 21,9 millions de sommets — on ne l'énumère pas, on le couvre : toute distance de jeu s'obtient en sommant deux distances de côté, dans les deux modèles. Le recensement du §4, exhaustif *par côté*, est donc exhaustif *pour les jeux* au sens du lemme. (La BFS produit ci-dessus n'est pas une preuve du lemme — elle en est un contrôle ; la preuve est l'argument : chaque pas touche un seul côté.)

Pourquoi garder ce contrôle coûteux alors que l'argument est simple ? Parce que le lemme porte tout le notebook : si l'entrelacement des générateurs cachait une subtilité — par exemple un swap simultané des deux côtés compté pour un pas, ou une interaction entre les murs de Ligne et ceux de Colonne —, toutes les distances de jeu dérivées du §4 seraient fausses *sans aucun signal d'erreur local*. Le contrôle direct sur l'antipode (le cas où l'erreur éventuelle serait maximale, 30 étant le diamètre) est le témoin qui aurait contredit le premier. C'est la même discipline que le vérificateur du §6 : on ne croit pas un argument élégant sans un recalcul indépendant là où il est le plus vulnérable.

In [9]:
// === Section 6.1 : construire_chemin -- le constructeur temoin (portage GT-20) ===

// Produit une suite (table, table) de depart a arrivee par BFS + remontee des parents.
static List<(int[] A, int[] B)> ConstruireChemin(int[] depart, int[] arrivee, Dictionary<string, List<int[]>> adj)
{
    var parent = new Dictionary<string, int[]> { [Key(depart)] = null };
    var q = new Queue<int[]>(new[] { depart });
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        if (Key(u) == Key(arrivee)) break;
        foreach (var v in adj[Key(u)])
        {
            var kv = Key(v);
            if (!parent.ContainsKey(kv))
            {
                parent[kv] = u;
                q.Enqueue(v);
            }
        }
    }
    if (!parent.ContainsKey(Key(arrivee))) return null;
    var chaine = new List<(int[], int[])>();
    var cur = arrivee;
    while (parent[Key(cur)] is not null)
    {
        var p = parent[Key(cur)];
        chaine.Add((p, cur));
        cur = p;
    }
    chaine.Reverse();
    return chaine;
}

var chemin_perm = ConstruireChemin(IDENTITE.Ligne, RENVERSE.Ligne, graphs[6].AdjP);
Console.WriteLine($"Chemin de chambres Identite -> Renversement (cote Ligne) : {chemin_perm.Count} pas");
for (int i = 0; i < 5; i++)
    Console.WriteLine($"  pas {i + 1,2} : {Tst(chemin_perm[i].A)} -> {Tst(chemin_perm[i].B)}");
Console.WriteLine("  ...");
var chemin_murs = ConstruireChemin(IDENTITE.Ligne, RENVERSE.Ligne, graphs[6].AdjC);
Console.WriteLine($"Chemin par les murs  Identite -> Renversement (cote Ligne) : {chemin_murs.Count} pas");
for (int i = 0; i < chemin_murs.Count; i++)
    Console.WriteLine($"  pas {i + 1,2} : {Tst(chemin_murs[i].A)} -> {Tst(chemin_murs[i].B)}");

Chemin de chambres Identite -> Renversement (cote Ligne) : 15 pas


  pas  1 : (1, 2, 3, 4, 5, 6) -> (2, 1, 3, 4, 5, 6)


  pas  2 : (2, 1, 3, 4, 5, 6) -> (3, 1, 2, 4, 5, 6)


  pas  3 : (3, 1, 2, 4, 5, 6) -> (3, 2, 1, 4, 5, 6)


  pas  4 : (3, 2, 1, 4, 5, 6) -> (4, 2, 1, 3, 5, 6)


  pas  5 : (4, 2, 1, 3, 5, 6) -> (4, 3, 1, 2, 5, 6)


... (8 outputs left out)

  pas  7 : (2, 2, 2, 2, 2, 1) -> (3, 3, 3, 3, 2, 1)


  pas  8 : (3, 3, 3, 3, 2, 1) -> (4, 4, 4, 3, 2, 1)


  pas  9 : (4, 4, 4, 3, 2, 1) -> (5, 5, 4, 3, 2, 1)


  pas 10 : (5, 5, 4, 3, 2, 1) -> (6, 5, 4, 3, 2, 1)


### Lecture du résultat — le témoin de l'opération 13 : 15 pas par les chambres, 10 par les murs, 5 fusions puis 5 scissions

Le constructeur produit les deux témoins de côté : 15 pas par les chambres, **10 pas par les murs** — et l'affichage du chemin de 10 pas révèle sa structure : les 5 premières étapes sont des **fusions** (la table se lie progressivement jusqu'à la table tout-liée `(1, 1, 1, 1, 1, 1)`), les 5 suivantes des **scissions** qui relâchent les blocs directement dans l'ordre renversé. Le chemin traverse la *cellule centrale* du treillis — le mur total — et c'est précisément cette traversée qui vaut 5 pas d'économie. C'est le témoin de l'opération 13 : un mur réellement traversé, dans les deux sens, produisant un gain mesuré.

In [10]:
// === Section 6.2 : verifier_chemin -- le verificateur independant (Loi II) ===
// Le verificateur ne reutilise NI l'adjacence du constructeur, NI ses structures : il re-ecrit
// ses propres predicats et refait une recherche exhaustive pour la preuve de minimalite.
// Deux modeles, deux verificateurs, chacun re-derive du zero.

// Re-derivation independante du voisinage du treillis (code separe de la section 2.1).
static List<int[]> VoisinsTreillisVerif(int[] t)
{
    var vals = t.Distinct().OrderBy(v => v).ToList();
    var output = new List<int[]>();
    for (int k = 1; k < 6; k++)
        if (vals.Contains(k) && vals.Contains(k + 1))
            output.Add(Canonique(t.Select(v => v == k + 1 ? k : v).ToArray()));
    foreach (var v in vals)
    {
        var pos = new List<int>();
        for (int i = 0; i < 6; i++) if (t[i] == v) pos.Add(i);
        if (pos.Count < 2) continue;
        for (int mask = 1; mask < (1 << pos.Count) - 1; mask++)
        {
            var bas = new HashSet<int>();
            for (int j = 0; j < pos.Count; j++)
                if ((mask >> j & 1) == 1) bas.Add(pos[j]);
            var nouv = new int[6];
            for (int i = 0; i < 6; i++)
            {
                if (bas.Contains(i)) nouv[i] = v;
                else if (t[i] == v) nouv[i] = v + 1;
                else nouv[i] = t[i] > v ? t[i] + 1 : t[i];
            }
            output.Add(Canonique(nouv));
        }
    }
    return output;
}

// Re-derivation independante du voisinage du permutoedre : echange des valeurs k, k+1.
static List<int[]> VoisinsPermVerif(int[] t)
{
    var output = new List<int[]>();
    for (int k = 1; k < 6; k++)
        output.Add(t.Select(v => v == k ? k + 1 : v == k + 1 ? k : v).ToArray());
    return output;
}

static Dictionary<string, int> BfsDepart(int[] src, Func<int[], List<int[]>> voisinsFn)
{
    var dist = new Dictionary<string, int> { [Key(src)] = 0 };
    var q = new Queue<int[]>(new[] { src });
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        foreach (var v in voisinsFn(u))
        {
            var kv = Key(v);
            if (!dist.ContainsKey(kv))
            {
                dist[kv] = dist[Key(u)] + 1;
                q.Enqueue(v);
            }
        }
    }
    return dist;
}

// Verdict independant : extremites, elementarite de chaque pas, minimalite exhaustive.
static string VerdictChemin(int[] depart, int[] arrivee, List<(int[] A, int[] B)> chaine,
                            Func<int[], List<int[]>> voisinsFn, string nomModele)
{
    if (chaine is null || chaine.Count == 0)
        return "INVALIDE : chemin vide";
    if (Key(chaine[0].A) != Key(depart) || Key(chaine[^1].B) != Key(arrivee))
        return "INVALIDE : les extremites ne sont pas celles annoncees";
    foreach (var (pred, cur) in chaine)
        if (!voisinsFn(pred).Any(v => Key(v) == Key(cur)))
            return $"INVALIDE : pas non elementaire {Tst(pred)} -> {Tst(cur)}";
    var dist = BfsDepart(depart, voisinsFn);
    if (chaine.Count != dist[Key(arrivee)])
        return $"INVALIDE : non minimal ({chaine.Count} pas, distance reelle {dist[Key(arrivee)]})";
    return $"VALIDE [{nomModele}] : {chaine.Count} pas, minimalite certifiee par recherche exhaustive independante";
}

Console.WriteLine("Chemin de chambres (15 pas) juge par le verificateur du permutoedre :");
Console.WriteLine($"  {VerdictChemin(IDENTITE.Ligne, RENVERSE.Ligne, chemin_perm, VoisinsPermVerif, "permutoedre")}");
Console.WriteLine("Chemin par les murs (10 pas) juge par le verificateur du treillis :");
Console.WriteLine($"  {VerdictChemin(IDENTITE.Ligne, RENVERSE.Ligne, chemin_murs, VoisinsTreillisVerif, "treillis")}");

Chemin de chambres (15 pas) juge par le verificateur du permutoedre :


  VALIDE [permutoedre] : 15 pas, minimalite certifiee par recherche exhaustive independante


Chemin par les murs (10 pas) juge par le verificateur du treillis :


  VALIDE [treillis] : 10 pas, minimalite certifiee par recherche exhaustive independante


### Lecture du résultat — deux minimalités dans deux graphes, certifiées par deux vérificateurs sans ligne commune

Chaque témoin est certifié **dans son propre modèle**, par un vérificateur qui ré-écrit tout : le chemin de 15 pas est validé élémentaire et minimal au sens des swaps (vérificateur du permutoèdre), le chemin de 10 pas au sens fusions/scissions (vérificateur du treillis) — comparez les deux implémentations de voisinage : elles ne partagent aucune ligne, c'est la séparation de la Loi II. Un point mérite d'être souligné : le chemin de chambres n'est **pas** un chemin du treillis — chacun de ses swaps s'y décompose en fusion + scission (§2.1). Les 15 et les 10 sont donc deux minimalités de deux graphes différents, et c'est précisément leur écart — 5 pas sur l'antipode — qui constitue la mesure de la contraction métrique du §4.

In [11]:
// === Section 7 : controle negatif -- le verificateur doit aussi savoir dire NON ===

var faux_compose = new List<(int[] A, int[] B)>
{
    (IDENTITE.Ligne, SwapAdjacent(SwapAdjacent(IDENTITE.Ligne, 1), 2)),
    (SwapAdjacent(SwapAdjacent(IDENTITE.Ligne, 1), 2), RENVERSE.Ligne),
};
var faux_extremites = new List<(int[] A, int[] B)> { (RENVERSE.Ligne, SwapAdjacent(RENVERSE.Ligne, 1)) };

// Detour valide mais non minimal : le chemin de murs (10 pas) rallonge d'un aller-retour
// sur le mur 5~6 (fusion puis scission retour), toujours elementaire -> 12 pas.
var x = Fusion(IDENTITE.Ligne, 5);
var chemin_detour = new List<(int[] A, int[] B)> { (IDENTITE.Ligne, x), (x, IDENTITE.Ligne) };
chemin_detour.AddRange(chemin_murs);

Console.WriteLine("Controle 1 -- pas compose de deux swaps declare au verificateur du treillis :");
Console.WriteLine($"  {VerdictChemin(IDENTITE.Ligne, RENVERSE.Ligne, faux_compose, VoisinsTreillisVerif, "treillis")}");
Console.WriteLine("Controle 2 -- extremites fausses :");
Console.WriteLine($"  {VerdictChemin(IDENTITE.Ligne, RENVERSE.Ligne, faux_extremites, VoisinsPermVerif, "permutoedre")}");
Console.WriteLine("Controle 3 -- detour de 12 pas (elementaire, extremites justes, non minimal) :");
Console.WriteLine($"  {VerdictChemin(IDENTITE.Ligne, RENVERSE.Ligne, chemin_detour, VoisinsTreillisVerif, "treillis")}");

Controle 1 -- pas compose de deux swaps declare au verificateur du treillis :


  INVALIDE : pas non elementaire (1, 2, 3, 4, 5, 6) -> (3, 1, 2, 4, 5, 6)


Controle 2 -- extremites fausses :


  INVALIDE : les extremites ne sont pas celles annoncees


Controle 3 -- detour de 12 pas (elementaire, extremites justes, non minimal) :


  INVALIDE : non minimal (12 pas, distance reelle 10)


### Lecture du résultat — le contrôle négatif : trois modes de refus, dont un détour de 12 pas jugé non minimal

Un vérificateur qui ne saurait que dire `VALIDE` ne certifierait rien. Les trois contrôles exhibent trois modes de refus distincts : le pas composé de deux swaps est rejeté comme **non élémentaire** (aucun voisinage ne contient ce saut double), les extrémités fausses sont détectées avant toute recherche, et — le contrôle qui fait le travail — un détour *parfaitement licite* de 12 pas (fusion vers le mur 5~6 puis scission retour, insérée dans le vrai chemin de 10) est détecté **non minimal**. C'est lui qui prouve que la preuve de minimalité a les dents : elle ne se contente pas de valider la connectivité, elle distingue 12 de 10.

In [12]:
// === Section 8 : geometriques identite -> renversement sur le permutoedre S6 ===

// Nombre de chemins minimaux : produits des chemins entrants, couche par couche (DAG des couches).
static long CompterGeodesiques(int[] depart, int[] arrivee, Dictionary<string, List<int[]>> adj)
{
    var dist = Bfs(depart, adj);
    int d = dist[Key(arrivee)];
    var nb = new Dictionary<string, long> { [Key(depart)] = 1 };
    for (int couche = 1; couche <= d; couche++)
        foreach (var (ku, du) in dist)
            if (du == couche)
                nb[ku] = adj[ku].Where(v => dist.GetValueOrDefault(Key(v)) == couche - 1)
                                .Sum(v => nb.GetValueOrDefault(Key(v)));
    return nb[Key(arrivee)];
}

var geo6 = CompterGeodesiques(IDENTITE.Ligne, RENVERSE.Ligne, graphs[6].AdjP);
Console.WriteLine($"Geodesiques {Tst(IDENTITE.Ligne)} -> {Tst(RENVERSE.Ligne)} sur une table 3x2 : {geo6}");
Console.WriteLine("Rappel GT-20 (S4, substrat 2x2) : 16 geodesiques pour l'antipode");

Geodesiques (1, 2, 3, 4, 5, 6) -> (6, 5, 4, 3, 2, 1) sur une table 3x2 : 292864


Rappel GT-20 (S4, substrat 2x2) : 16 geodesiques pour l'antipode


### Lecture du résultat — 292 864 géodésiques au lieu de 16 : l'explosion du nombre de chemins minimaux

L'explosion du nombre de géodésiques (16 sur S4, **292 864** sur S6 — la valeur mesurée ci-dessus) matérialise la richesse du nouveau substrat : il n'existe pas *un* chemin minimal mais une famille entière, et le diamètre 15 n'est réalisé que par une fraction organisée des 720 tables. Ce comptage par DAG des couches est le même instrument que GT-20 (cross-check direct avec ses 16), porté à l'échelle supérieure sans changement de méthode — la reproductibilité de l'instrument à travers les substrats fait partie de l'attestation.

Le nombre de géodésiques dit aussi quelque chose de la *robustesse* du témoin du §6 : si l'antipode n'était atteint que par un unique chemin de 10 pas, on pourrait soupçonner une coincidence structurelle ; le fait qu'il existe par ailleurs des familles entières de chemins minimaux dans chaque modèle rappelle que la minimalité certifiée par le vérificateur porte sur la *longueur*, et que le constructeur n'en a extrait qu'un représentant. Un prolongement naturel (hors scope ici) serait de compter les géodésiques du trellis entre antipodes — la version « à travers les murs » de ce comptage — pour comparer les deux cardinaux.

In [13]:
// === Exercice 1 : chemin retour Renversement -> Identite, construit puis verifie independamment. ===
// Etape 1 : var cheminRetour = ConstruireChemin(RENVERSE.Ligne, IDENTITE.Ligne, graphs[6].AdjC);
// Etape 2 : var verdictRetour = VerdictChemin(RENVERSE.Ligne, IDENTITE.Ligne, cheminRetour, VoisinsTreillisVerif, "treillis");
// Indice : le treillis est non dirige -- la longueur du retour doit egaler 10.
List<(int[] A, int[] B)>? chemin_retour = null;  // TODO etudiant
string? verdict_retour = null;  // TODO etudiant
Console.WriteLine("Exercice 1 : a completer (chemin retour Renversement -> Identite par les murs)");

Exercice 1 : a completer (chemin retour Renversement -> Identite par les murs)



(5,25): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(6,7): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Exercice 1 — Le chemin retour

Construisez le chemin de retour `Renversement -> Identité` sur le graphe du treillis, puis faites-le certifier par le vérificateur. *Indice :* le graphe est non dirigé, donc la longueur du retour est connue d'avance — c'est exactement ce que le vérificateur doit confirmer par sa propre recherche exhaustive.

In [14]:
// === Exercice 2 : les antipodes raccourcissent-ils tous ? ===
// Etape 1 : var dperm = Bfs(IDENTITE.Ligne, graphs[6].AdjP); var antipodes = ST6.Where(t => dperm[Key(t)] == 15).ToList();
// Etape 2 : var dtl = Bfs(IDENTITE.Ligne, graphs[6].AdjC); compter combien d'antipodes ont dtl[Key(t)] < 15.
// Indice : le chemin tout-lie (5 fusions + 5 scissions = 10) n'est pas la seule facon de gagner.
int? antipodes_raccourcis = null;  // TODO etudiant
Console.WriteLine("Exercice 2 : a completer (combien des antipodes de distance 15 sont raccourcis par les murs)");

Exercice 2 : a completer (combien des antipodes de distance 15 sont raccourcis par les murs)


### Exercice 2 — Le recensement des antipodes

Parmi les tables situées à la distance maximale 15 de l'identité dans le permutoèdre (les antipodes), combien sont raccourcies par les murs ? *Indice :* le chemin tout-lié en vaut 5 d'économie, mais certaines réorganisations partielles peuvent faire mieux ou moins bien — la distribution des gains est le sujet.

In [15]:
// === Exercice 3 : traverser un mur triple. ===
// Etape 1 : var cible = Canonique(new[] { 2, 2, 2, 1, 3, 4 });  // trois cases liees au rang 2
// Etape 2 : var chemin = ConstruireChemin(IDENTITE.Ligne, cible, graphs[6].AdjC);
// Etape 3 : var verdict = VerdictChemin(IDENTITE.Ligne, cible, chemin, VoisinsTreillisVerif, "treillis");
// Indice : combien de fusions pour lier trois cases, et la scission finale est-elle utile ici ?
List<(int[] A, int[] B)>? mur_triple = null;  // TODO etudiant
Console.WriteLine("Exercice 3 : a completer (chemin minimal vers une table portant un ex aequo triple)");

Exercice 3 : a completer (chemin minimal vers une table portant un ex aequo triple)



(6,25): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Exercice 3 — Le mur triple

Construisez et certifiez un chemin minimal de l'identité vers une table portant un ex æquo **triple**. *Indice :* lier trois cases demande deux fusions ; demandez-vous si le vérificateur accepte un chemin qui *re-descend* scissionner quelque part — la réponse mesure la frontière entre réordonner par les murs et payer les allers-retours.

## Conclusion — ce que cette seconde attestation établit

**Pour la table des opérations [#12204](https://github.com/jsboige/CoursIA/issues/12204), opération 13 « Traverser un mur »** : le témoin GT-20 (2×2) n'était pas isolé dans son genre — sa méthode (constructeur/vérificateur séparés, recensement exhaustif, contrôle négatif) se transpose à un second substrat, et c'est précisément cette transplantation qui produit la connaissance :

1. **Le verdict 2×2 est robuste au modèle fermé** : 0 raccourci sur 576 paires, murs traversables dans les deux sens (§3). La mesure GT-20 n'était pas un artefact du sens unique.
2. **Le verdict ne survit pas au substrat 3×2** : 77 760 raccourcis sur 518 400 paires (15,0 %), diamètre par côté contracté de 15 à 10 (§4). Traverser les murs y est un raccourci *structurel* — porté par les scissions de blocs — et non une curiosité marginale.
3. **Le témoin est certifié** : chemin de l'antipode à travers le mur total (5 fusions + 5 scissions), construit puis validé par un vérificateur qui re-dérive tout (§6), contrôle négatif à l'appui (§7).

**Résiduel assumé.** L'univers complet de jeux avec murs bilatéraux (≈ 21,9 M sommets) est couvert par le lemme de décomposition (§5), contrôlé mais pas prouvé formellement ici ; une preuve Lean du lemme serait le prolongement naturel côté `game_theory_lean`.

**Généalogie** : [Metajeux-04](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) (témoin constructeur/vérificateur, 2×2) · [Metajeux-05](Metajeux-05-Temoins-Impossibilite-Python.ipynb) (témoins d'impossibilité) · ce notebook (second substrat, verdict de frontière).

**Retour à la série.** [GameTheory — README](../README.md) — Metajeux-06 est le dernier carnet de la sous-série `Metajeux`.

## À retenir

- **Le verdict 2×2 ne survit PAS au substrat 3×2** : ici, les murs *raccourcissent* — `77 760` raccourcis sur `518 400` paires (`15.0%`), et le diamètre par côté est contracté de 15 à 10. Traverser les murs y est un raccourci *structurel* (porté par les scissions de blocs), pas une curiosité marginale.
- **Le témoin est certifié par transplantation de méthode** : le constructeur produit le chemin de l'antipode à travers le mur total (5 fusions + 5 scissions), et un vérificateur *re-dérive tout* pour attester sa minimalité — le même instrument que GT-20, porté à l'échelle supérieure.
- **La richesse du nouveau substrat s'explore en nombre de chemins minimaux** : `292 864` géodésiques pour l'antipode 3×2, contre 16 sur `S₄` — le comptage par DAG des couches est le même instrument, sans changement de méthode.

## Vérifiez votre compréhension

1. Pourquoi le verdict « 0 raccourci » de GT-20 (2×2) ne se transpose-t-il PAS au substrat 3×2 ? Qu'est-ce que les scissions de blocs changent à la géométrie ?
2. En quoi le comptage `292 864` (contre 16) atteste-t-il de la *robustesse* du témoin plutôt que d'une coïncidence structurelle ?
3. Que couvre le lemme de décomposition (§5) pour l'univers complet à murs bilatéraux (≈ 21,9 M sommets), et pourquoi sa preuve Lean reste-t-elle un résiduel assumé ?

<details>
<summary>Éléments de réponse</summary>

1. Parce que le troc n'a pas la même arithmétique aux deux tailles. Une traversée de mur coûte au moins deux pas — une fusion, une scission — et ne peut être rentable que si le déplacement de blocs économise au moins deux swaps ; renverser n cases coûte `C(n,2)` swaps au permutoèdre contre `2(n-1)` pas par les murs, d'où un gain net nul à 4 cases et de 5 à 6 cases. À 2×2, la mesure le confirme : `0` raccourci sur `576` paires, `48` égalités. À 3×2, une seule scission d'un bloc de taille m replace m cases d'un coup là où le permutoèdre les permute une à une : le recensement exhaustif donne `77760` raccourcis sur `518400` paires (`15.0 %`), le chemin de l'antipode passant de `15` swaps à `10` pas — les cinq premières étapes fusionnent, les cinq suivantes scissionnent — et le diamètre par côté se contractant de `15` à `10`. La scission change donc la géométrie : elle rend le déplacement massif possible en un pas, ce qui transforme une stratification métriquement neutre en compression métrique.
2. Parce que le même instrument redonne un compte sur le nouveau substrat sans changer de méthode : `292864` géodésiques pour l'antipode sur une table 3×2, contre `16` sur `S₄` rappelé dans la même sortie. Le témoin du §6 n'est donc pas un chemin unique qu'un accident aurait produit : sur le permutoèdre, l'antipode est atteint par une famille entière de chemins minimaux, et la propriété certifiée porte sur la **longueur** — le vérificateur re-dérive la distance exhaustive et rend `VALIDE [permutoedre] : 15 pas` et `VALIDE [treillis] : 10 pas`, et le contrôle négatif distingue `12` de `10` —, pas sur l'unicité de l'itinéraire. Le constructeur n'extrait qu'un représentant de la famille ; ce qui rend le témoin robuste, c'est que la minimalité vaut pour toute la famille et que l'instrument de comptage se reproduit d'un substrat à l'autre. Le diamètre mesuré `15` rejoint d'ailleurs la prédiction de Coxeter (inversions de `w₀` dans `S₆`), ce qui écarte l'hypothèse d'une coïncidence du substrat.
3. Le lemme `d_jeu(G,H) = d_cote(row_G,row_H) + d_cote(col_G,col_H)`, valable dans les deux modèles, couvre l'univers des jeux à murs des deux côtés — `4683` ordres faibles par côté, soit `4683² ≈ 21,9 millions` de sommets — sans l'énumérer : chaque générateur ne touche qu'un seul côté, donc toute distance de jeu se décompose en deux distances de côté. C'est ce qui rend le recensement du §4, mené sur les seules paires de chambres, exhaustif **pour les jeux** au sens du lemme. Le contrôle direct existe : la BFS sur le produit des `518400` chambres atteint `518400` jeux et donne, identité → renversement, `directe = 30 | par decomposition = 30`, `concordance : True` — mais un contrôle n'est pas une preuve ; la preuve est l'argument « un pas touche un seul côté », contrôlé là où une erreur serait maximale (l'antipode, distance `30`, le diamètre). Sa formalisation Lean n'est pas faite ici : le carnet l'assume comme résiduel et la renvoie au prolongement naturel côté `game_theory_lean`.

</details>

## Pour aller plus loin

- **La généalogie de la série** : [Metajeux-04](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) (témoin 2×2, murs non-raccourcissants) · [Metajeux-05](Metajeux-05-Temoins-Impossibilite-Python.ipynb) (témoins d'impossibilité) · ce notebook (second substrat, murs raccourcissants).
- **Le lemme de décomposition en Lean** : le prolongement naturel côté `game_theory_lean` pour prouver formellement la couverture de l'univers complet.